# Notebook 53 — Field+Year Normalised Target

**Hypothesis**: The global 75th-percentile threshold (≥26 citations) conflates citation-count norms that differ across research fields. A paper in Medicine accumulates citations differently from one in Engineering. Redefining "high impact" as the top 25% *within the same ASJC field and publication year* should produce a cleaner, more consistent label — and may improve F1 on both AUB-only and merged data.

**Prior art**:
- `nb41b` tried **year-only** normalisation → **−3.86 pp** F1. Interpretation: temporal bias was not limiting performance.
- `nb48` tried **per-institution** thresholds → recovered to ≈ clean baseline (51.28%) on merged data.
- `nb51` revealed the original 62.55% baseline was inflated by +8.14 pp (leakage). True clean AUB-only baseline: **51.28%**.

**Field+year normalisation differs from year-only** because it corrects for fundamentally different citation cultures across disciplines, not just temporal accumulation.

## Configurations

| Config | Label scheme | Test thresholds from | Training data | TF-IDF fit on |
|--------|-------------|---------------------|--------------|---------------|
| REF-CLEAN | Global threshold | train | AUB long window | AUB train |
| A | Field-only threshold | train | AUB long window | AUB train |
| B | Field+year threshold | **test set** (CNCI-style) | AUB long window | AUB train |
| B* | Field+year threshold | **train only** (ablation) | AUB long window | AUB train |
| C | Field+year threshold | test set | All institutions (per-inst) | merged train |
| D | Field+year threshold | test set | All institutions (per-inst) | **AUB-only** train |

**Config B vs B* ablation**: B uses test-set citation distributions to set test thresholds (valid, analogous to CNCI); B* uses only training thresholds for test labels. If B* F1 ≈ B F1, the improvement is genuine. If B* drops back toward REF-CLEAN, part of the F1 gain in B comes from the test distribution enforcing a cleaner ~25% positive rate rather than from better discriminative power. Use **AUC** as the primary discriminative signal — it is threshold- and class-balance-independent.

In [ ]:
import sys
sys.path.append('../../')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, roc_auc_score, precision_score, recall_score

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
TRAIN_YEARS  = list(range(2010, 2018))   # long window: 2010-2017
TEST_YEARS   = [2018, 2019, 2020]
QUANTILE     = 0.75                      # top-25% = high impact

# Reference values from nb51
NB51_CLEAN_BASELINE = 0.5128   # REF-CLEAN AUB-only, long window
NB48_BEST_F1        = 0.6253   # per-inst labels + inst features (on leaked baseline — use cautiously)

print('Libraries loaded')

## 1. Load data

In [ ]:
data_path = Path('../../data/processed/all_unis_cleaned.pkl')

if not data_path.exists():
    raise FileNotFoundError(
        f"Merged data not found: {data_path}\n"
        "Run notebooks 04 → 05 → 06 first to generate the merged dataset."
    )

df = pd.read_pickle(data_path)

print(f"Loaded: {df.shape[0]:,} rows × {df.shape[1]} cols")
print(f"\nInstitution counts:")
print(df['institution'].value_counts().to_string())
print(f"\nYear range: {df['Year'].min()} – {df['Year'].max()}")
print(f"\nCitation statistics:")
print(df['Citations'].describe().to_string())

In [ ]:
# Detect ASJC column
if 'All Science Journal Classification (ASJC) field name' in df.columns:
    ASJC_COL = 'All Science Journal Classification (ASJC) field name'
elif 'ASJC field name' in df.columns:
    ASJC_COL = 'ASJC field name'
else:
    candidates = [c for c in df.columns if 'asjc' in c.lower()]
    ASJC_COL = candidates[0] if candidates else None

print(f"ASJC column: '{ASJC_COL}'")

if ASJC_COL:
    print(f"\nTop 20 ASJC fields:")
    print(df[ASJC_COL].value_counts().head(20).to_string())
    print(f"\nTotal unique fields: {df[ASJC_COL].nunique()}")
    print(f"Missing ASJC data: {df[ASJC_COL].isna().sum()} / {len(df)} ({df[ASJC_COL].isna().mean():.1%})")
else:
    print("WARNING: No ASJC column found. Field-normalisation will not be possible.")

In [ ]:
# Temporal splits
df_aub        = df[df['institution'] == 'AUB'].copy()
df_train      = df[df['Year'].isin(TRAIN_YEARS)].copy()
df_test       = df[df['Year'].isin(TEST_YEARS)].copy()
df_test_aub   = df_test[df_test['institution'] == 'AUB'].copy()
df_aub_train  = df_aub[df_aub['Year'].isin(TRAIN_YEARS)].copy()

INSTS = sorted(df['institution'].unique())

print(f"All-inst train  (2010-2017): {len(df_train):,}")
print(f"AUB-only train  (2010-2017): {len(df_aub_train):,}")
print(f"Test            (2018-2020): {len(df_test):,}")
print(f"AUB test        (2018-2020): {len(df_test_aub):,}")
print(f"\nInstitutions: {INSTS}")

## 2. Helper functions (feature building, evaluation)

In [ ]:
COL_MAP = {
    'snip':             'SNIP (publication year)',
    'snip_pct':         'SNIP percentile',
    'citescore':        'CiteScore (publication year)',
    'citescore_pct':    'CiteScore percentile',
    'sjr':              'SJR (publication year)',
    'sjr_pct':          'SJR percentile',
    'topic_prom':       'Topic Prominence Percentile',
    'num_authors':      'Authors',
    'num_institutions': 'Affiliations',
    'num_countries':    'Countries',
}


def extract_venue_features(subset_df):
    vf = pd.DataFrame(index=subset_df.index)
    for feat, col in COL_MAP.items():
        if col in subset_df.columns:
            if col in ('Authors', 'Affiliations', 'Countries'):
                vf[feat] = subset_df[col].fillna('').apply(
                    lambda x: len(str(x).split(';')) if x else 1
                )
            else:
                vf[feat] = pd.to_numeric(subset_df[col], errors='coerce')
    return vf


def build_tfidf(df_tr, df_te):
    """Fit TF-IDF on df_tr abstracts, transform both sets."""
    tfidf = TfidfVectorizer(
        max_features=5000, ngram_range=(1, 2),
        min_df=5, max_df=0.8, stop_words='english'
    )
    prep = lambda s: str(s).lower() if pd.notna(s) else ''
    tr_mat = tfidf.fit_transform(df_tr['Abstract'].apply(prep))
    te_mat = tfidf.transform(df_te['Abstract'].apply(prep))
    cols   = [f'tfidf_{f}' for f in tfidf.get_feature_names_out()]
    return (
        pd.DataFrame(tr_mat.toarray(), index=df_tr.index, columns=cols),
        pd.DataFrame(te_mat.toarray(), index=df_te.index, columns=cols)
    )


def build_features(df_tr, df_te):
    """Build X_train / X_test with TF-IDF + venue/author features."""
    tfidf_tr, tfidf_te = build_tfidf(df_tr, df_te)
    vf_tr = extract_venue_features(df_tr)
    vf_te = extract_venue_features(df_te)
    tr_median = vf_tr.median()
    vf_tr = vf_tr.fillna(tr_median)
    vf_te = vf_te.fillna(tr_median)
    X_tr = pd.concat([tfidf_tr, vf_tr.set_index(tfidf_tr.index)], axis=1)
    X_te = pd.concat([tfidf_te, vf_te.set_index(tfidf_te.index)], axis=1)
    return X_tr, X_te


def evaluate(model, X_tr, y_tr, X_te, y_te, label=''):
    """Fit model, grid-search threshold, return metrics dict."""
    model.fit(X_tr, y_tr)
    proba = model.predict_proba(X_te)[:, 1]
    thresholds = np.arange(0.10, 0.91, 0.01)
    f1s    = [f1_score(y_te, (proba >= t).astype(int), zero_division=0) for t in thresholds]
    best_t = thresholds[int(np.argmax(f1s))]
    y_pred = (proba >= best_t).astype(int)
    return {
        'label':     label,
        'f1':        f1_score(y_te, y_pred, zero_division=0),
        'auc':       roc_auc_score(y_te, proba),
        'recall':    recall_score(y_te, y_pred, zero_division=0),
        'precision': precision_score(y_te, y_pred, zero_division=0),
        'threshold': best_t,
        'n_train':   len(y_tr),
        'n_test':    len(y_te),
        'pos_rate_train': float(y_tr.mean()),
        'pos_rate_test':  float(y_te.mean()),
    }


MODEL = LogisticRegression(
    max_iter=1000, class_weight='balanced',
    C=1.0, solver='lbfgs', random_state=RANDOM_STATE
)

results = []
print('Helpers ready.')

## 3. Label-generation functions

In [ ]:
# ── Label helpers ─────────────────────────────────────────────────────────────

def global_labels(df_tr, df_te, quantile=QUANTILE):
    """Single threshold computed on training set only (no leakage)."""
    thr = df_tr['Citations'].quantile(quantile)
    y_tr = (df_tr['Citations'] >= thr).astype(int)
    y_te = (df_te['Citations'] >= thr).astype(int)
    print(f"  Global threshold: {thr:.0f}  |  train pos: {y_tr.mean():.1%}  |  test pos: {y_te.mean():.1%}")
    return y_tr, y_te


def field_labels(df_tr, df_te, asjc_col, quantile=QUANTILE, fallback='global'):
    """
    Top-`quantile` within each ASJC field.
    Thresholds computed from training data; applied to test.
    Papers in unseen field fall back to the global training threshold.
    """
    global_thr = df_tr['Citations'].quantile(quantile)
    field_thr  = (
        df_tr.groupby(asjc_col)['Citations']
             .quantile(quantile)
             .rename('thr')
    )

    def assign(row):
        field = row[asjc_col]
        thr   = field_thr.get(field, global_thr) if pd.notna(field) else global_thr
        return int(row['Citations'] >= thr)

    y_tr = df_tr.apply(assign, axis=1)
    y_te = df_te.apply(assign, axis=1)

    n_fields  = df_tr[asjc_col].nunique()
    n_covered = df_te[asjc_col].isin(field_thr.index).mean()
    print(f"  Fields in training: {n_fields}  |  test coverage: {n_covered:.1%}")
    print(f"  train pos: {y_tr.mean():.1%}  |  test pos: {y_te.mean():.1%}")
    return y_tr, y_te


def field_year_labels(df_tr, df_te, asjc_col, quantile=QUANTILE):
    """
    Top-`quantile` within (ASJC field, Year).

    Train thresholds: computed from training papers' (field, year).
    Test thresholds:
      - Primary: (field, year) from test set itself — valid, no train leakage.
      - Fallback for missing: field-only training threshold, then global training threshold.

    Rationale for test-set threshold from test data: we are defining what counts as
    high impact for 2018-2020 papers using the citation distribution of 2018-2020 papers.
    This is analogous to year-normalisation used in bibliometrics (CNCI).
    """
    global_thr_tr   = df_tr['Citations'].quantile(quantile)
    field_thr_tr    = df_tr.groupby(asjc_col)['Citations'].quantile(quantile).rename('thr')
    fy_thr_tr       = df_tr.groupby([asjc_col, 'Year'])['Citations'].quantile(quantile).rename('thr')
    fy_thr_te       = df_te.groupby([asjc_col, 'Year'])['Citations'].quantile(quantile).rename('thr')

    def _label(row, fy_thr, field_thr, global_thr):
        field = row[asjc_col]
        year  = row['Year']
        key   = (field, year)
        if pd.isna(field):
            thr = global_thr
        elif key in fy_thr.index:
            thr = fy_thr[key]
        elif field in field_thr.index:
            thr = field_thr[field]
        else:
            thr = global_thr
        return int(row['Citations'] >= thr)

    y_tr = df_tr.apply(_label, axis=1, fy_thr=fy_thr_tr,
                       field_thr=field_thr_tr, global_thr=global_thr_tr)
    y_te = df_te.apply(_label, axis=1, fy_thr=fy_thr_te,
                       field_thr=field_thr_tr, global_thr=global_thr_tr)

    n_fy_tr = df_tr.groupby([asjc_col, 'Year']).ngroups
    n_fy_te = df_te.groupby([asjc_col, 'Year']).ngroups
    print(f"  (field, year) groups — train: {n_fy_tr}  |  test: {n_fy_te}")
    print(f"  train pos: {y_tr.mean():.1%}  |  test pos: {y_te.mean():.1%}")
    return y_tr, y_te


def field_year_labels_train_only(df_tr, df_te, asjc_col, quantile=QUANTILE):
    """
    ABLATION for Config B*: field+year normalisation using ONLY training-derived
    thresholds for both train and test labels.

    Test papers whose (field, year) combo was unseen during training fall back to
    the field-only training threshold, then the global training threshold.

    Purpose: isolate the 'label-cleaning effect' of field+year normalisation from
    the 'test-distribution effect' in Config B.
      - If B* F1 ≈ B F1  → the improvement is real (better-defined labels help).
      - If B* F1 ≈ REF-CLEAN F1  → Config B's gain came largely from the test set
        self-defining its positive rate at ~25%, not from genuine discrimination.
    AUC (which is threshold- and balance-independent) is the cleaner signal here.
    """
    global_thr_tr = df_tr['Citations'].quantile(quantile)
    field_thr_tr  = df_tr.groupby(asjc_col)['Citations'].quantile(quantile).rename('thr')
    fy_thr_tr     = df_tr.groupby([asjc_col, 'Year'])['Citations'].quantile(quantile).rename('thr')

    def _label(row):
        field = row[asjc_col]
        year  = row['Year']
        key   = (field, year)
        if pd.isna(field):
            thr = global_thr_tr
        elif key in fy_thr_tr.index:
            thr = fy_thr_tr[key]
        elif field in field_thr_tr.index:
            thr = field_thr_tr[field]
        else:
            thr = global_thr_tr
        return int(row['Citations'] >= thr)

    y_tr = df_tr.apply(_label, axis=1)
    y_te = df_te.apply(_label, axis=1)

    # Report test coverage: what fraction of test (field,year) pairs had a training threshold
    n_covered_fy = df_te.apply(
        lambda r: pd.notna(r[asjc_col]) and (r[asjc_col], r['Year']) in fy_thr_tr.index,
        axis=1
    ).mean()
    n_fy_tr = df_tr.groupby([asjc_col, 'Year']).ngroups
    print(f"  (field, year) groups in train: {n_fy_tr}")
    print(f"  Test papers covered by train (field,year) threshold: {n_covered_fy:.1%}")
    print(f"  train pos: {y_tr.mean():.1%}  |  test pos: {y_te.mean():.1%}")
    return y_tr, y_te


def per_inst_field_year_labels(df_tr, df_te_aub, asjc_col, quantile=QUANTILE):
    """
    Per-institution field+year thresholds for training data;
    AUB-specific field+year thresholds for test data.
    """
    global_thr_tr = df_tr['Citations'].quantile(quantile)

    # Compute per-(institution, field, year) training thresholds
    inst_fy_thr = (
        df_tr.groupby(['institution', asjc_col, 'Year'])['Citations']
             .quantile(quantile)
             .rename('thr')
    )
    # Fallback: per-(institution, field)
    inst_field_thr = (
        df_tr.groupby(['institution', asjc_col])['Citations']
             .quantile(quantile)
             .rename('thr')
    )
    # AUB field+year thresholds for test (from test data itself)
    aub_fy_thr_te = (
        df_te_aub.groupby([asjc_col, 'Year'])['Citations']
                 .quantile(quantile)
                 .rename('thr')
    )
    aub_field_thr_tr = inst_field_thr.xs('AUB', level='institution') if 'AUB' in inst_field_thr.index.get_level_values('institution') else pd.Series(dtype=float)

    def _label_tr(row):
        inst  = row['institution']
        field = row[asjc_col]
        year  = row['Year']
        if pd.isna(field):
            return int(row['Citations'] >= global_thr_tr)
        for key, idx in [((inst, field, year), inst_fy_thr), ((inst, field), inst_field_thr)]:
            if key in idx.index:
                return int(row['Citations'] >= idx[key])
        return int(row['Citations'] >= global_thr_tr)

    def _label_te(row):
        field = row[asjc_col]
        year  = row['Year']
        if pd.isna(field):
            return int(row['Citations'] >= global_thr_tr)
        key_fy = (field, year)
        if key_fy in aub_fy_thr_te.index:
            return int(row['Citations'] >= aub_fy_thr_te[key_fy])
        if field in aub_field_thr_tr.index:
            return int(row['Citations'] >= aub_field_thr_tr[field])
        return int(row['Citations'] >= global_thr_tr)

    y_tr = df_tr.apply(_label_tr, axis=1)
    y_te = df_te_aub.apply(_label_te, axis=1)

    print(f"  train pos: {y_tr.mean():.1%}  |  test pos: {y_te.mean():.1%}")
    return y_tr, y_te


print('Label helpers ready.')

## 4. REF-CLEAN — Global threshold, AUB-only data (reproduces nb51)

In [ ]:
print("=" * 60)
print("REF-CLEAN: Global threshold, AUB-only, long window")
print("=" * 60)

y_tr_ref, y_te_ref = global_labels(df_aub_train, df_test_aub)

X_tr_ref, X_te_ref = build_features(df_aub_train, df_test_aub)

import copy
res = evaluate(copy.deepcopy(MODEL), X_tr_ref, y_tr_ref, X_te_ref, y_te_ref,
               label='REF-CLEAN (global, AUB-only)')
results.append(res)

print(f"\n  F1: {res['f1']:.4f}  |  AUC: {res['auc']:.4f}")
print(f"  Recall: {res['recall']:.4f}  |  Precision: {res['precision']:.4f}")
print(f"  Expected from nb51: {NB51_CLEAN_BASELINE:.4f}")

## 5. Config A — Field-only threshold, AUB data

In [ ]:
print("=" * 60)
print("Config A: Field-only threshold, AUB-only, long window")
print("=" * 60)

if ASJC_COL is None:
    print("SKIPPED — ASJC column not found.")
else:
    y_tr_a, y_te_a = field_labels(df_aub_train, df_test_aub, ASJC_COL)

    X_tr_a, X_te_a = build_features(df_aub_train, df_test_aub)

    res_a = evaluate(copy.deepcopy(MODEL), X_tr_a, y_tr_a, X_te_a, y_te_a,
                     label='Config A (field-only, AUB-only)')
    results.append(res_a)

    delta = res_a['f1'] - results[0]['f1']
    print(f"\n  F1: {res_a['f1']:.4f}  ({delta:+.4f} vs REF-CLEAN)")
    print(f"  AUC: {res_a['auc']:.4f}")
    print(f"  Recall: {res_a['recall']:.4f}  |  Precision: {res_a['precision']:.4f}")

## 6. Config B — Field+year threshold, AUB data

In [ ]:
print("=" * 60)
print("Config B: Field+year threshold, AUB-only, long window")
print("=" * 60)

if ASJC_COL is None:
    print("SKIPPED — ASJC column not found.")
else:
    y_tr_b, y_te_b = field_year_labels(df_aub_train, df_test_aub, ASJC_COL)

    # Features same as REF-CLEAN (AUB-only TF-IDF)
    X_tr_b, X_te_b = X_tr_ref, X_te_ref

    res_b = evaluate(copy.deepcopy(MODEL), X_tr_b, y_tr_b, X_te_b, y_te_b,
                     label='Config B (field+year, AUB-only)')
    results.append(res_b)

    delta = res_b['f1'] - results[0]['f1']
    print(f"\n  F1: {res_b['f1']:.4f}  ({delta:+.4f} vs REF-CLEAN)")
    print(f"  AUC: {res_b['auc']:.4f}")
    print(f"  Recall: {res_b['recall']:.4f}  |  Precision: {res_b['precision']:.4f}")

## 6b. Config B* — Field+year, train-only thresholds (ablation)

Same label scheme as Config B but test labels are assigned using **only training-derived thresholds** — test papers in unseen (field, year) combos fall back to field-only then global training thresholds.

This separates two possible explanations for Config B's F1 gain:
- **Label-cleaning effect**: field+year normalisation removes cross-field citation-culture noise, making labels more consistent → model trains on cleaner signal
- **Test-distribution effect**: using test-set thresholds forces the test positive rate toward ~25% per group, which can mechanically improve F1 even without better discrimination

AUC is the decisive metric here: it does not depend on the classification threshold or class balance.

In [ ]:
print("=" * 60)
print("Config B*: Field+year threshold, train-only thresholds, AUB-only")
print("=" * 60)

if ASJC_COL is None:
    print("SKIPPED — ASJC column not found.")
else:
    y_tr_bstar, y_te_bstar = field_year_labels_train_only(df_aub_train, df_test_aub, ASJC_COL)

    # Same features as Config B / REF-CLEAN
    X_tr_bstar, X_te_bstar = X_tr_ref, X_te_ref

    res_bstar = evaluate(copy.deepcopy(MODEL), X_tr_bstar, y_tr_bstar, X_te_bstar, y_te_bstar,
                         label='Config B* (field+year, train-only thr)')
    results.append(res_bstar)

    delta_vs_ref = res_bstar['f1'] - results[0]['f1']
    delta_vs_b   = res_bstar['f1'] - res_b['f1']
    delta_auc_vs_ref = res_bstar['auc'] - results[0]['auc']
    delta_auc_vs_b   = res_bstar['auc'] - res_b['auc']

    print(f"\n  F1:  {res_bstar['f1']:.4f}  ({delta_vs_ref:+.4f} vs REF-CLEAN  |  {delta_vs_b:+.4f} vs Config B)")
    print(f"  AUC: {res_bstar['auc']:.4f}  ({delta_auc_vs_ref:+.4f} vs REF-CLEAN  |  {delta_auc_vs_b:+.4f} vs Config B)")
    print(f"  Recall: {res_bstar['recall']:.4f}  |  Precision: {res_bstar['precision']:.4f}")
    print(f"  pos_rate_test: {res_bstar['pos_rate_test']:.1%}  (Config B: {res_b['pos_rate_test']:.1%}  REF: {results[0]['pos_rate_test']:.1%})")

    print("\n  --- Interpretation ---")
    f1_gap_b_bstar = abs(res_b['f1'] - res_bstar['f1'])
    auc_gap_b_bstar = res_b['auc'] - results[0]['auc']  # Config B AUC gain vs REF
    if f1_gap_b_bstar < 0.02:
        print("  B* F1 ≈ B F1 → F1 gain is robust to threshold source; label cleaning is real.")
    elif res_bstar['f1'] < results[0]['f1'] + 0.01:
        print("  B* F1 ≈ REF-CLEAN → Config B F1 gain is largely a test-distribution artifact.")
    else:
        print("  B* is intermediate → mixed effect (some real improvement, some distributional).")

    if auc_gap_b_bstar > 0.01:
        print(f"  AUC gain of {auc_gap_b_bstar:+.4f} in Config B → discriminative improvement is genuine.")

## 7. Config C — Field+year threshold, merged data, per-institution

In [ ]:
print("=" * 60)
print("Config C: Field+year threshold, merged train, per-inst labels")
print("=" * 60)

if ASJC_COL is None:
    print("SKIPPED — ASJC column not found.")
else:
    y_tr_c, y_te_c = per_inst_field_year_labels(df_train, df_test_aub, ASJC_COL)

    # TF-IDF fit on merged train data (same as nb51 Config E/F/G)
    X_tr_c, X_te_c = build_features(df_train, df_test_aub)

    res_c = evaluate(copy.deepcopy(MODEL), X_tr_c, y_tr_c, X_te_c, y_te_c,
                     label='Config C (field+year, merged, per-inst)')
    results.append(res_c)

    delta = res_c['f1'] - results[0]['f1']
    print(f"\n  F1: {res_c['f1']:.4f}  ({delta:+.4f} vs REF-CLEAN)")
    print(f"  AUC: {res_c['auc']:.4f}")
    print(f"  Recall: {res_c['recall']:.4f}  |  Precision: {res_c['precision']:.4f}")

## 8. Config D — Field+year threshold, merged data, AUB-only TF-IDF

In [ ]:
res_df = pd.DataFrame(results)
ref_f1  = res_df.loc[res_df['label'].str.startswith('REF'), 'f1'].values[0]
ref_auc = res_df.loc[res_df['label'].str.startswith('REF'), 'auc'].values[0]
res_df['delta_f1_vs_ref']  = res_df['f1']  - ref_f1
res_df['delta_auc_vs_ref'] = res_df['auc'] - ref_auc

display_cols = ['label', 'f1', 'delta_f1_vs_ref', 'auc', 'delta_auc_vs_ref',
                'recall', 'precision', 'threshold', 'pos_rate_train', 'pos_rate_test']
display_cols = [c for c in display_cols if c in res_df.columns]

print("\n" + "=" * 100)
print("RESULTS SUMMARY")
print("=" * 100)
print(res_df[display_cols].to_string(index=False, float_format='{:.4f}'.format))
print(f"\nnb51 clean baseline reference: F1={NB51_CLEAN_BASELINE:.4f}")

# ── Key diagnostic: B vs B* ────────────────────────────────────────────────────
if 'Config B*' in ' '.join(res_df['label'].tolist()):
    row_b     = res_df[res_df['label'].str.startswith('Config B (')].iloc[0]
    row_bstar = res_df[res_df['label'].str.startswith('Config B*')].iloc[0]
    row_ref   = res_df[res_df['label'].str.startswith('REF')].iloc[0]

    print("\n" + "=" * 60)
    print("KEY DIAGNOSTIC: Config B vs B* (ablation)")
    print("=" * 60)
    print(f"  REF-CLEAN  F1={row_ref['f1']:.4f}  AUC={row_ref['auc']:.4f}  pos_rate_test={row_ref['pos_rate_test']:.1%}")
    print(f"  Config B   F1={row_b['f1']:.4f}  AUC={row_b['auc']:.4f}  pos_rate_test={row_b['pos_rate_test']:.1%}")
    print(f"  Config B*  F1={row_bstar['f1']:.4f}  AUC={row_bstar['auc']:.4f}  pos_rate_test={row_bstar['pos_rate_test']:.1%}")
    print(f"\n  F1  gap B vs B*:   {row_b['f1']  - row_bstar['f1']:+.4f}")
    print(f"  AUC gap B vs REF:  {row_b['auc']  - row_ref['auc']:+.4f}   ← threshold-independent signal")
    print(f"  AUC gap B* vs REF: {row_bstar['auc'] - row_ref['auc']:+.4f}   ← same model, train-only thresholds")
    print(f"\n  NOTE: AUC is the primary discriminative signal. If AUC gain holds in B*,")
    print(f"  field+year-normalised TRAINING labels genuinely improve the model.")
    print(f"  If F1 gap (B vs B*) is large but AUC gap (B vs REF) is small,")
    print(f"  the F1 improvement is a class-balance artefact, not real discrimination.")

## 9. Results summary

In [ ]:
res_df = pd.DataFrame(results)
ref_f1 = res_df.loc[res_df['label'].str.startswith('REF'), 'f1'].values[0]
res_df['delta_vs_ref'] = res_df['f1'] - ref_f1

display_cols = ['label', 'f1', 'delta_vs_ref', 'auc', 'recall', 'precision',
                'threshold', 'n_train', 'pos_rate_train', 'pos_rate_test']
display_cols = [c for c in display_cols if c in res_df.columns]

print("\n" + "=" * 80)
print("RESULTS SUMMARY")
print("=" * 80)
print(res_df[display_cols].to_string(index=False, float_format='{:.4f}'.format))
print(f"\nnb51 clean baseline reference: {NB51_CLEAN_BASELINE:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

colors = ['#4C8BE2' if d >= 0 else '#E24C4C' for d in res_df['delta_vs_ref']]
labels = [l.split('(')[0].strip() for l in res_df['label']]

# F1
axes[0].barh(labels, res_df['f1'], color=colors)
axes[0].axvline(NB51_CLEAN_BASELINE, color='red', linestyle='--', label='nb51 baseline')
axes[0].set_title('F1 Score')
axes[0].legend(fontsize=8)
for i, (v, d) in enumerate(zip(res_df['f1'], res_df['delta_vs_ref'])):
    axes[0].text(v + 0.001, i, f'{v:.4f} ({d:+.4f})', va='center', fontsize=8)

# AUC
axes[1].barh(labels, res_df['auc'], color='#4C8BE2')
axes[1].set_title('ROC-AUC')
for i, v in enumerate(res_df['auc']):
    axes[1].text(v + 0.001, i, f'{v:.4f}', va='center', fontsize=8)

# Precision vs Recall
axes[2].scatter(res_df['recall'], res_df['precision'], s=80, color='#4C8BE2', zorder=3)
for _, row in res_df.iterrows():
    lbl = row['label'].split('(')[0].strip()
    axes[2].annotate(lbl, (row['recall'], row['precision']),
                     textcoords='offset points', xytext=(5, 2), fontsize=7)
axes[2].set_xlabel('Recall')
axes[2].set_ylabel('Precision')
axes[2].set_title('Precision vs Recall')
axes[2].grid(True, alpha=0.3)

plt.suptitle('Notebook 53 — Field+Year Normalised Target: Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('../../docs/nb53_field_year_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Plot saved to docs/nb53_field_year_results.png')

## 10. Field-level analysis — where does normalisation change labels?

In [ ]:
print("=" * 70)
print("NOTEBOOK 53 — CONCLUSIONS")
print("=" * 70)

# Ensure delta columns exist (in case cell-21 was skipped)
if 'delta_f1_vs_ref' not in res_df.columns:
    ref_f1  = res_df.loc[res_df['label'].str.startswith('REF'), 'f1'].values[0]
    ref_auc = res_df.loc[res_df['label'].str.startswith('REF'), 'auc'].values[0]
    res_df['delta_f1_vs_ref']  = res_df['f1']  - ref_f1
    res_df['delta_auc_vs_ref'] = res_df['auc'] - ref_auc

best = res_df.loc[res_df['f1'].idxmax()]
print(f"\nBest config by F1: {best['label']}")
print(f"  F1:  {best['f1']:.4f}  ({best['delta_f1_vs_ref']:+.4f} vs REF-CLEAN)")
print(f"  AUC: {best['auc']:.4f}  ({best['delta_auc_vs_ref']:+.4f} vs REF-CLEAN)")
print(f"  Recall: {best['recall']:.4f}  |  Precision: {best['precision']:.4f}")
print(f"\nnb51 clean baseline: {NB51_CLEAN_BASELINE:.4f}")
print(f"Overall F1 delta vs nb51: {best['f1'] - NB51_CLEAN_BASELINE:+.4f}")

print("\n--- Summary table ---")
for _, row in res_df.iterrows():
    f1_sign   = '+' if row['delta_f1_vs_ref'] >= 0 else ''
    auc_sign  = '+' if row['delta_auc_vs_ref'] >= 0 else ''
    f1_verdict = 'IMPROVED' if row['delta_f1_vs_ref'] > 0.005 else ('DEGRADED' if row['delta_f1_vs_ref'] < -0.005 else 'FLAT')
    print(f"  {f1_verdict:8s}  {row['label']:52s}"
          f"  F1={row['f1']:.4f} ({f1_sign}{row['delta_f1_vs_ref']:.4f})"
          f"  AUC={row['auc']:.4f} ({auc_sign}{row['delta_auc_vs_ref']:.4f})"
          f"  pos_test={row['pos_rate_test']:.1%}")

# ── B vs B* verdict ────────────────────────────────────────────────────────────
has_bstar = res_df['label'].str.startswith('Config B*').any()
has_b     = res_df['label'].str.startswith('Config B (').any()
if has_b and has_bstar:
    row_b     = res_df[res_df['label'].str.startswith('Config B (')].iloc[0]
    row_bstar = res_df[res_df['label'].str.startswith('Config B*')].iloc[0]
    row_ref   = res_df[res_df['label'].str.startswith('REF')].iloc[0]

    f1_gap    = row_b['f1']  - row_bstar['f1']
    auc_b     = row_b['auc']  - row_ref['auc']
    auc_bstar = row_bstar['auc'] - row_ref['auc']

    print("\n" + "=" * 70)
    print("B vs B* ABLATION VERDICT")
    print("=" * 70)
    if abs(f1_gap) < 0.02:
        print(f"  F1 gap B–B* = {f1_gap:+.4f} (< 2pp) → label-cleaning is the primary driver.")
        print(f"  Field+year normalisation of TRAINING labels genuinely reduces noise.")
    elif row_bstar['f1'] < row_ref['f1'] + 0.01:
        print(f"  F1 gap B–B* = {f1_gap:+.4f} → Config B gain is largely a test-distribution artefact.")
        print(f"  Recommend reporting AUC as the primary metric, not F1.")
    else:
        print(f"  F1 gap B–B* = {f1_gap:+.4f} → mixed: partial real improvement, partial distributional.")

    print(f"\n  AUC Δ (Config B  vs REF-CLEAN): {auc_b:+.4f}")
    print(f"  AUC Δ (Config B* vs REF-CLEAN): {auc_bstar:+.4f}")
    if auc_bstar > 0.005:
        print(f"  → Training on field+year labels improves discriminative power (AUC up in B*).")
    else:
        print(f"  → No AUC gain in B* → field+year normalisation on training labels does not help.")

## 11. Conclusions


In [ ]:
print("=" * 70)
print("NOTEBOOK 53 — CONCLUSIONS")
print("=" * 70)

best = res_df.loc[res_df['f1'].idxmax()]
print(f"\nBest config: {best['label']}")
print(f"  F1:  {best['f1']:.4f}  ({best['delta_vs_ref']:+.4f} vs REF-CLEAN)")
print(f"  AUC: {best['auc']:.4f}")
print(f"  Recall: {best['recall']:.4f}  |  Precision: {best['precision']:.4f}")
print(f"\nnb51 clean baseline: {NB51_CLEAN_BASELINE:.4f}")
print(f"Overall delta vs nb51: {best['f1'] - NB51_CLEAN_BASELINE:+.4f}")

print("\n--- Summary table ---")
for _, row in res_df.iterrows():
    sign = '+' if row['delta_vs_ref'] >= 0 else ''
    verdict = 'IMPROVED' if row['delta_vs_ref'] > 0.005 else ('DEGRADED' if row['delta_vs_ref'] < -0.005 else 'FLAT')
    print(f"  {verdict:8s}  {row['label']:50s}  F1={row['f1']:.4f}  ({sign}{row['delta_vs_ref']:.4f})")